# MicroCLIP — Colab A100 training & evaluation

Reproduces the MicroCLIP study on a Colab **A100**: trains any config from the
ablation matrix, mirrors checkpoints to Drive without filling Trash, and evaluates
every run on COCO 5K retrieval + zero-shot CIFAR.

**Setup:** Runtime → Change runtime type → **A100 GPU**, then run the cells top to bottom.

### Storage layout (Drive-space friendly)

| What | Where | Size |
|---|---|---|
| COCO images | local SSD (`/content`) | ~20 GB, never touches Drive |
| Checkpoints while training | local SSD (`runs/`) | written every 500 steps + each epoch |
| Checkpoint mirror | Drive `.../runs/<run>/` | `last_a.pt` + `last_b.pt` (~224 MB each) + `best.pt` (~75 MB) ≈ **0.5 GB/run while training** |
| After a run | keep `best.pt` (~75 MB); drop `last_a/b` to reclaim | eval only needs `best.pt` |
| Tokenizer | Drive `.../artifacts/` | ~1 MB |

**Why not write checkpoints straight to Drive:** the trainer replaces the ~224 MB
`last.pt` dozens of times per run. Files replaced/deleted through the Colab Drive
mount go to Drive **Trash**, which still counts against quota — one run can fill a
15 GB Drive. Instead a background process (`ckpt_sync.py`) copies checkpoints to
Drive every few minutes, overwriting two fixed slot files **in place** (no deletes
→ no Trash). If a copy is torn by preemption, the other slot still holds a valid
checkpoint. `best.pt` is also flushed explicitly at the end of each run.

**Preemption:** re-run the training cell. The newest readable Drive slot is
restored to local disk and training auto-resumes (losing at most one sync interval).
The queue is idempotent — finished `(config, seed)` runs are skipped.


In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > A100"
name = torch.cuda.get_device_name(0)
print(name, "| bf16:", torch.cuda.is_bf16_supported())
if "A100" not in name:
    print("WARNING: not an A100 — configs assume Ampere+ (bf16 AMP, batch 512)")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
PERSIST = "/content/drive/MyDrive/microclip"
os.makedirs(f"{PERSIST}/runs", exist_ok=True)
os.makedirs(f"{PERSIST}/artifacts/tokenizer", exist_ok=True)

# Budget ~0.5 GB of free Drive per run kept while training. If Drive is near full,
# empty Drive Trash first (drive.google.com -> Trash) — it counts against quota.
!df -h /content/drive | tail -1
!du -sh {PERSIST} 2>/dev/null || true


In [ ]:
%cd /content
!git clone https://github.com/umutonuryasar/microclip.git 2>/dev/null || git -C microclip pull
%cd /content/microclip
!pip -q install -e .

# pip -e registers src/ via a .pth file, which a running kernel never re-reads —
# without this, `import microclip` fails until a runtime restart.
import sys
if "/content/microclip/src" not in sys.path:
    sys.path.insert(0, "/content/microclip/src")


In [ ]:
import os

# Tokenizer is tiny -> lives on Drive so it is trained once.
if not os.path.islink("artifacts"):
    os.system("rm -rf artifacts")
    os.symlink(f"{PERSIST}/artifacts", "artifacts")

# Checkpoints go to the LOCAL SSD (fast, no Drive Trash churn); the sync process
# mirrors them to Drive. Undo any old runs/ -> Drive symlink.
if os.path.islink("runs"):
    os.unlink("runs")
os.makedirs("runs", exist_ok=True)
print("artifacts/ ->", os.readlink("artifacts"))
print("runs/ is local:", os.path.abspath("runs"))


In [ ]:
%%bash
# COCO 2017 to local SSD (~20 GB unzipped, ~40 GB peak while unzipping train2017).
df -h /content | tail -1
mkdir -p data/coco && cd data/coco
for z in train2017 val2017; do
  if [ ! -d $z ]; then
    wget -q -c http://images.cocodataset.org/zips/$z.zip
    unzip -q $z.zip && rm $z.zip
  fi
done
if [ ! -d annotations ]; then
  wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -q annotations_trainval2017.zip && rm annotations_trainval2017.zip
fi
du -sh *


In [ ]:
import os

# One-off; lands on Drive via the artifacts/ symlink.
if not os.path.exists("artifacts/tokenizer/bpe16k.json"):
    !python scripts/train_tokenizer.py --config configs/base.yml
else:
    print("tokenizer exists — skipping")


In [ ]:
import os
from google.colab import userdata

# Store your key once: Colab left sidebar -> key icon -> add WANDB_API_KEY.
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import wandb
wandb.login()
# train.py calls wandb.init(id=run_name, resume="allow") + wandb.finish(), so a
# resumed run continues the same W&B run and finishes cleanly (no "crashed" rows).


### Optional pre-flight: smoke test
Exercises the real data path for a few steps on both losses before committing GPU hours. Safe to skip on later sessions of the same clone.


In [ ]:
# ~few min on A100; writes only to local runs/smoke_*, cleaned up afterwards.
!python scripts/smoke_test.py --config configs/base.yml && rm -rf runs/smoke_*


## Checkpoint sync (Drive-safe)

The next cell writes `ckpt_sync.py`: a background process that mirrors local
`runs/<run>/{last,best}.pt` to Drive, alternating two fixed slot files
(`last_a.pt` / `last_b.pt`) so no delete/rename ever hits Drive Trash. The training
cell starts and stops it automatically per run. Run this **once per session**.


In [ ]:
%%writefile /content/ckpt_sync.py
"""Mirror runs/<run>/{last,best}.pt to Drive without filling Drive Trash.

last.pt alternates between two fixed slot files (last_a.pt / last_b.pt) and every
copy overwrites a slot in place -- no delete/rename on Drive, so no Trash growth.
A copy torn by preemption only damages one slot; the other stays valid. best.pt is
mirrored to a single fixed file. On stop, one final sync pass runs before exit.
"""
import os
import shutil
import sys
import time

local, remote, interval, stop = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]
os.makedirs(remote, exist_ok=True)
slots = [os.path.join(remote, "last_a.pt"), os.path.join(remote, "last_b.pt")]
mtime = lambda p: os.path.getmtime(p) if os.path.exists(p) else 0.0
slot = 0 if mtime(slots[0]) <= mtime(slots[1]) else 1  # overwrite the older slot first
seen = {}


def push(name):
    global slot
    src = os.path.join(local, name)
    if not os.path.exists(src):
        return
    # Open first: the trainer's os.replace() can't swap the file under us mid-copy.
    with open(src, "rb") as fsrc:
        m = os.fstat(fsrc.fileno()).st_mtime
        if seen.get(name) == m:
            return
        dst = slots[slot] if name == "last.pt" else os.path.join(remote, name)
        with open(dst, "wb") as fdst:  # truncate + rewrite the same Drive file
            shutil.copyfileobj(fsrc, fdst, 16 << 20)
    seen[name] = m
    if name == "last.pt":
        slot ^= 1  # flip only after a successful copy
    print(f"[sync {time.strftime('%H:%M:%S')}] {name} -> {os.path.basename(dst)}", flush=True)


while True:
    stopping = os.path.exists(stop)
    for name in ("best.pt", "last.pt"):
        try:
            push(name)
        except Exception as e:
            print(f"[sync] {name} failed: {e}", flush=True)
    if stopping:
        os.remove(stop)
        print("[sync] final sync done, exiting", flush=True)
        break
    for _ in range(interval):  # responsive to the stop file
        if os.path.exists(stop):
            break
        time.sleep(1)


## Training — seed-matrix queue

One idempotent queue drives all training. Each job is `(config, seed, run_name, overrides)`.
For every job the queue:

1. restores the newest Drive checkpoint to local disk if resuming on a fresh VM,
2. starts `ckpt_sync.py` in the background,
3. runs `scripts/train.py` (seeds + `run_name` passed via `--set`),
4. on finish, stops the sync and **explicitly flushes** local `best.pt`/`last.pt` to
   Drive (the background sync can lag by one interval; this guarantees `best.pt` is
   the true final checkpoint),
5. skips any `(config, seed)` already finished on Drive.

The **main matrix** is b512 + b128 × sigmoid/softmax × 3 seeds = 12 runs (~17 h).
Single-seed extras (b256, the 10-epoch anchor, and the ablations) are listed
commented-out below — uncomment what you want to (re)run.

Set `SHUTDOWN_WHEN_DONE = True` to release the VM when the queue finishes.
Run **evaluation in a fresh session** afterwards.


In [ ]:
import os, re, shutil, subprocess, time, torch
from microclip.config import load_config

SEEDS              = [42, 43, 44]
SYNC_EVERY_MIN     = 15
SHUTDOWN_WHEN_DONE = False          # True: release the VM after the queue finishes
STOP = "/content/ckpt_sync.stop"
LOG  = f"{PERSIST}/queue_log.txt"

# --- job list: (config, seed, run_name, extra_overrides) ---------------------
MAIN = [
    "configs/sigmoid_b512.yml",
    "configs/softmax_b512.yml",
    "configs/ablations/sigmoid_b128.yml",
    "configs/ablations/softmax_b128.yml",
]
jobs = []
for cfg_path in MAIN:                       # 3-seed main matrix
    base = load_config(cfg_path)["run_name"]
    for s in SEEDS:
        jobs.append((cfg_path, s, f"{base}_s{s}", ""))

# Single-seed extras — uncomment to (re)produce b256, the anchor, and ablations:
# jobs += [
#     ("configs/ablations/sigmoid_b256.yml", 42, "sigmoid_b256",    ""),
#     ("configs/ablations/softmax_b256.yml", 42, "softmax_b256",    ""),
#     ("configs/base.yml",                   42, "base_s42_ep10",   "train.epochs=10"),
#     ("configs/ablations/optimizer_sgd.yml",42, "abl_sgd",         ""),
#     ("configs/ablations/lr_constant.yml",  42, "abl_lr_constant", ""),
#     ("configs/ablations/init_xavier.yml",  42, "abl_init_xavier", ""),
#     ("configs/ablations/init_he.yml",      42, "abl_init_he",     ""),
#     ("configs/ablations/vit_tiny.yml",     42, "abl_vit_tiny",    ""),
# ]

# --- helpers -----------------------------------------------------------------
def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG, "a") as f:
        f.write(line + "\n")

def newest_slot(remote):
    """Path of the resume checkpoint with the highest global_step, or None."""
    best = None
    for cand in ("last_a.pt", "last_b.pt", "last.pt"):
        p = f"{remote}/{cand}"
        if not os.path.exists(p):
            continue
        try:
            s = torch.load(p, map_location="cpu", weights_only=False)
        except Exception:
            continue
        if best is None or s["global_step"] > best[0]:
            best = (s["global_step"], s["epoch"], p)
    return best

def target_epochs(cfg_path, overrides):
    m = re.search(r"train\.epochs=(\d+)", overrides)
    return int(m.group(1)) if m else load_config(cfg_path)["train"]["epochs"]

def stop_sync(proc):
    if proc and proc.poll() is None:
        open(STOP, "w").close()
        proc.wait()
    if os.path.exists(STOP):
        os.remove(STOP)

# --- run one job -------------------------------------------------------------
def run_job(cfg_path, seed, run_name, overrides):
    total = target_epochs(cfg_path, overrides)
    local, remote = f"runs/{run_name}", f"{PERSIST}/runs/{run_name}"
    os.makedirs(local, exist_ok=True)
    os.makedirs(remote, exist_ok=True)

    prog = newest_slot(remote)
    if prog and prog[1] >= total:
        log(f"{run_name}: already finished (epoch {prog[1]}) — skip")
        return
    if not os.path.exists(f"{local}/last.pt") and prog:                 # resume
        shutil.copyfile(prog[2], f"{local}/last.pt")
        log(f"{run_name}: restored step {prog[0]} from Drive")
    if os.path.exists(f"{remote}/best.pt") and not os.path.exists(f"{local}/best.pt"):
        shutil.copyfile(f"{remote}/best.pt", f"{local}/best.pt")

    if os.path.exists(STOP):
        os.remove(STOP)
    proc = subprocess.Popen(
        ["python", "/content/ckpt_sync.py", local, remote, str(SYNC_EVERY_MIN * 60), STOP],
        stdout=open("/content/ckpt_sync.log", "a"), stderr=subprocess.STDOUT)

    sets = f"data.num_workers=8 seed={seed} run_name={run_name}"
    if overrides:
        sets += f" {overrides}"
    log(f"{run_name}: training ({cfg_path}, seed={seed})")
    get_ipython().system(f"python scripts/train.py --config {cfg_path} --set {sets}")
    code = get_ipython().user_ns.get("_exit_code", 0)

    stop_sync(proc)                                                     # final sync + exit
    for fn, dst in (("best.pt", "best.pt"), ("last.pt", "last_a.pt")):  # explicit flush
        if os.path.exists(f"{local}/{fn}"):
            shutil.copyfile(f"{local}/{fn}", f"{remote}/{dst}")
    log(f"{run_name}: {'done' if code == 0 else f'FAILED (exit {code}) — moving on'}")

# --- drive the queue ---------------------------------------------------------
stop_sync(None)
for job in jobs:
    run_job(*job)
log("queue finished")

if SHUTDOWN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()


## Verify runs completed
Reads the newest resume slot of every run and prints its epoch / step / best val loss. Do this before evaluating so no half-finished run slips into the results.


In [ ]:
import os, torch
RUNS = f"{PERSIST}/runs"

for run in sorted(os.listdir(RUNS)):
    d = f"{RUNS}/{run}"
    if not os.path.isdir(d):
        continue
    slot = next((f"{d}/{c}" for c in ("last_a.pt", "last_b.pt", "last.pt")
                 if os.path.exists(f"{d}/{c}")), None)
    has_best = os.path.exists(f"{d}/best.pt")
    if slot is None:
        print(f"{run:20s} no resume checkpoint  best.pt={has_best}")
        continue
    s = torch.load(slot, map_location="cpu", weights_only=False)
    print(f"{run:20s} epoch={s['epoch']:>3} step={s['global_step']:>6} "
          f"best_val={s['best_val']:.4f}  best.pt={has_best}")


## Evaluation — seed-grouped (mean ± std)

Run this in a **fresh session** (re-run setup cells 1–6 first; W&B and the smoke
test are not needed). It evaluates every run's `best.pt` on COCO 5K retrieval and
zero-shot CIFAR, groups seeds by stripping the `_s<NN>` suffix, and reports
mean ± sample std. Single-seed groups report `nan` std. Writes `eval_per_run.csv`
and `eval_summary.csv` to Drive.


In [ ]:
import os, re, torch, numpy as np, pandas as pd
from torchvision.datasets import CIFAR10, CIFAR100
from microclip.config import load_config
from microclip.data.coco_captions import CocoCaptions
from microclip.data.tokenizer import CaptionTokenizer
from microclip.data.transforms import build_transforms
from microclip.eval.retrieval import encode_corpus, recall_at_k
from microclip.eval.zeroshot import zeroshot_accuracy
from microclip.models.microclip import MicroCLIP

RUNS = f"{PERSIST}/runs"
DEV  = "cuda"

# run folder (after stripping _s<NN>) -> the config it was trained with
CFG_BY_BASE = {
    "sigmoid_b512": "configs/sigmoid_b512.yml",
    "softmax_b512": "configs/softmax_b512.yml",
    "sigmoid_b256": "configs/ablations/sigmoid_b256.yml",
    "softmax_b256": "configs/ablations/softmax_b256.yml",
    "sigmoid_b128": "configs/ablations/sigmoid_b128.yml",
    "softmax_b128": "configs/ablations/softmax_b128.yml",
    "abl_vit_tiny": "configs/ablations/vit_tiny.yml",
    "abl_sgd": "configs/ablations/optimizer_sgd.yml",
    "abl_lr_constant": "configs/ablations/lr_constant.yml",
    "abl_init_xavier": "configs/ablations/init_xavier.yml",
    "abl_init_he": "configs/ablations/init_he.yml",
    "base_ep10": "configs/base.yml",
}

tf = build_transforms(224, train=False)
cifar = {"cifar10":  CIFAR10("data/cifar",  train=False, download=True, transform=tf),
         "cifar100": CIFAR100("data/cifar", train=False, download=True, transform=tf)}

per_run = {}
for run in sorted(os.listdir(RUNS)):
    d = f"{RUNS}/{run}"
    if not os.path.isdir(d) or not os.path.exists(f"{d}/best.pt"):
        continue
    base = re.sub(r"_s\d+", "", run)                 # sigmoid_b512_s43 -> sigmoid_b512
    cfg_path = CFG_BY_BASE.get(base)
    if cfg_path is None:
        print(f"skip {run}: no config mapping for '{base}'")
        continue
    cfg = load_config(cfg_path); dd = cfg["data"]
    tok = CaptionTokenizer(cfg["tokenizer"]["path"])
    model = MicroCLIP(cfg, vocab_size=tok.vocab_size, max_len=dd["max_text_len"])
    st = torch.load(f"{d}/best.pt", map_location="cpu", weights_only=False)
    model.load_state_dict(st["model"] if "model" in st else st)
    model = model.to(DEV).eval()

    ds = CocoCaptions(dd["root"], dd["val_images"], dd["val_ann"], tok,
                      dd["image_size"], dd["max_text_len"], train=False)
    im, tx, t2i = encode_corpus(model, ds, cfg, device=DEV)
    row = dict(recall_at_k(im, tx, t2i))
    for nm, cds in cifar.items():
        row[f"{nm}/top1"] = float(zeroshot_accuracy(model, tok, cds, cds.classes, cfg, device=DEV))
    per_run[run] = {k: float(v) for k, v in row.items()}
    print(run, {k: round(v, 4) for k, v in per_run[run].items()})
    del model, im, tx
    torch.cuda.empty_cache()

# group seeds -> mean +/- sample std (nan std for single-seed groups)
groups = {}
for run, row in per_run.items():
    groups.setdefault(re.sub(r"_s\d+", "", run), []).append(row)

metrics = ["t2i/R@1", "t2i/R@5", "t2i/R@10", "i2t/R@1", "i2t/R@5", "i2t/R@10",
           "cifar10/top1", "cifar100/top1"]
summary = {}
for base, rows in groups.items():
    n = len(rows)
    summary[base] = {"n": n}
    for m in metrics:
        v = np.array([r[m] for r in rows if m in r]) * 100
        sd = f"{v.std(ddof=1):.2f}" if n > 1 else "nan"
        summary[base][m] = f"{v.mean():.2f} +/- {sd}"

df = pd.DataFrame(summary).T
pd.DataFrame(per_run).T.to_csv(f"{PERSIST}/eval_per_run.csv")
df.to_csv(f"{PERSIST}/eval_summary.csv")
print("\n", df.to_string())


## Optional utilities


In [ ]:
# Reclaim Drive space: drop resume checkpoints (last_a/last_b/last) of FINISHED
# runs, keep best.pt (all eval needs). Then EMPTY DRIVE TRASH or space won't return.
import os
RUNS = f"{PERSIST}/runs"
freed = 0
for run in sorted(os.listdir(RUNS)):
    d = f"{RUNS}/{run}"
    if not os.path.isdir(d) or not os.path.exists(f"{d}/best.pt"):
        continue                       # only touch runs that have a best.pt (finished)
    for fn in ("last_a.pt", "last_b.pt", "last.pt"):
        p = f"{d}/{fn}"
        if os.path.exists(p):
            freed += os.path.getsize(p)
            os.remove(p)
print(f"freed ~{freed/1e9:.2f} GB (now in Drive Trash)")
print("IMPORTANT: drive.google.com -> Trash -> Empty trash, or the space stays used.")


In [ ]:
# Sanity-check that a checkpoint loads strictly and is the expected model.
import torch
from microclip.config import load_config
from microclip.data.tokenizer import CaptionTokenizer
from microclip.models.microclip import MicroCLIP

CONFIG = "configs/softmax_b512.yml"
CKPT   = f"{PERSIST}/runs/softmax_b512_s42/best.pt"   # canonical checkpoint

cfg = load_config(CONFIG); d = cfg["data"]
tok = CaptionTokenizer(cfg["tokenizer"]["path"])
model = MicroCLIP(cfg, vocab_size=tok.vocab_size, max_len=d["max_text_len"])
sd = torch.load(CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(sd["model"] if "model" in sd else sd, strict=True)
model.eval()
print("loaded OK:", CKPT)
print("logit_scale.exp() =", round(model.logit_scale.exp().item(), 2), "(~18 for a trained softmax run)")
